# 04. Pułapki i ograniczenia

**Poziom: Rozszerzenie dla dociekliwych**


## Spis treści

- [Problem](#sekcja-01)
- [Po tej części potrafisz](#sekcja-02)
- [Wykonanie](#sekcja-03)
- [1. `monostate` i powtarzające się typy](#sekcja-04)
    - [Jak zapisać „jeszcze nic nie wybrano”?](#sekcja-05)
    - [Gdy typ wymaga argumentu konstruktora](#sekcja-06)
    - [Czy ten sam typ może wystąpić dwa razy?](#sekcja-07)
- [2. Co oznacza `valueless_by_exception()`?](#sekcja-08)
    - [Jak wymuszamy ten przypadek w demonstracji?](#sekcja-09)
    - [Co można wtedy zrobić?](#sekcja-10)
- [3. Skąd wariant wie, który typ wybrać?](#sekcja-11)
    - [Dlaczego w przykładach tekst zapisujemy jawnie?](#sekcja-12)
- [4. Ile pamięci zajmuje wariant i czy można go kopiować?](#sekcja-13)
    - [Miejsce musi wystarczyć na każdy dozwolony typ](#sekcja-14)
    - [Czy wariant używa pamięci dynamicznej?](#sekcja-15)
    - [Od czego zależy kopiowanie?](#sekcja-16)
    - [Demonstracja: Pusty stan, wyjątek i rozmiar wariantu](#sekcja-17)
- [5. Kiedy wybrać inne rozwiązanie i czego uczyć się dalej?](#sekcja-18)
    - [Dobierz narzędzie do tego, co chcesz opisać](#sekcja-19)
    - [Co zmieniają nowsze standardy?](#sekcja-20)
    - [Dalszy temat: drzewo wyrażeń](#sekcja-21)
- [Dobre praktyki do zapamiętania](#sekcja-22)
- [Dodatek: rozpoznawanie błędów kompilacji](#sekcja-23)
- [Artykuły i dokumentacja](#sekcja-24)
    - [Artykuły z przykładami](#sekcja-25)
    - [Dokumentacja konkretnych operacji](#sekcja-26)


<a id="sekcja-01"></a>

## Problem

Typ opisuje dozwolone możliwości, ale trzeba jeszcze rozumieć inicjalizację, wyjątki i czas życia wartości. W tej części odróżnimy brak danych od utraty wartości po wyjątku i sprawdzimy dwa celowo błędne programy.

<a id="sekcja-02"></a>

## Po tej części potrafisz

- Odróżnić monostate od valueless_by_exception i rozpoznać błędy dostępu.
- Wyjaśnić wybór alternatywy, koszt pamięci i ograniczenia kopiowania.
- Rozpoznać dwa błędy kompilacji oraz dobrać variant lub inne narzędzie do problemu.

<a id="sekcja-03"></a>

## Wykonanie

Wybierz kernel Python 3 i wykonaj komórki od góry. Poniższa komórka Python przygotowuje `%%cpp`; każda komórka C++ zawiera własne nagłówki, definicje i `main()`. Nie wymaga uruchomienia pozostałych rozdziałów. W razie problemu wróć do [sprawdzenia środowiska](00_start.ipynb).

Dwie końcowe komórki używają `%%cpp --expect-error`: odrzucenie ich kodu przez kompilator jest oczekiwanym wynikiem.


In [1]:
import os
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.core.magic import register_cell_magic

COMPILER_NAME = os.environ.get("CXX", "c++")

@register_cell_magic
def cpp(line: str, cell: str) -> None:
    """Kompiluje i uruchamia jedną niezależną demonstrację C++17."""
    option = line.strip()
    if option not in {"", "--expect-error"}:
        raise ValueError("Użyj %%cpp albo %%cpp --expect-error.")
    expect_error = option == "--expect-error"
    compiler = shutil.which(COMPILER_NAME)
    if compiler is None:
        raise RuntimeError(
            f"Nie znaleziono kompilatora {COMPILER_NAME!r}. "
            "Zainstaluj kompilator C++17 lub ustaw COMPILER_NAME."
        )
    with TemporaryDirectory(prefix="variant-notebook-") as directory:
        source = Path(directory) / "example.cpp"
        executable = Path(directory) / ("example.exe" if os.name == "nt" else "example")
        source.write_text(cell, encoding="utf-8")
        compiled = subprocess.run(
            [compiler, "-std=c++17", "-Wall", "-Wextra", "-Wpedantic", "-Werror",
             str(source), "-o", str(executable)],
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=60,
        )
        if expect_error:
            if compiled.returncode == 0:
                raise RuntimeError("Oczekiwano błędu kompilacji, ale program skompilował się poprawnie.")
            print("Oczekiwany błąd kompilacji. Początek komunikatu kompilatora:")
            diagnostics = (compiled.stdout + compiled.stderr).replace(str(source), "example.cpp").splitlines()
            print("\n".join(diagnostics[:16]))
            if len(diagnostics) > 16:
                print("[Dalsza część komunikatu została pominięta.]")
            return
        if compiled.returncode != 0:
            raise RuntimeError("Błąd kompilacji:\n" + compiled.stdout + compiled.stderr)
        executed = subprocess.run(
            [str(executable)], capture_output=True, text=True,
            encoding="utf-8", errors="replace", timeout=10,
        )
        print(executed.stdout, end="")
        if executed.stderr:
            print(executed.stderr, end="")
        if executed.returncode != 0:
            raise RuntimeError(f"Program zakończył się kodem {executed.returncode}.")

print(f"Gotowe: %%cpp używa {COMPILER_NAME}, standard C++17.")

Gotowe: %%cpp używa c++, standard C++17.


<a id="sekcja-04"></a>

## 1. `monostate` i powtarzające się typy

<a id="sekcja-05"></a>

### Jak zapisać „jeszcze nic nie wybrano”?

Przypomnijmy: `std::variant<int, std::string> wartosc;` zaczyna od liczby 0. Czasem potrzebujesz jednak odróżnić „użytkownik podał zero” od „użytkownik jeszcze nic nie podał”.

Możesz dodać `std::monostate` jako pierwszą możliwość:


In [2]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

int main() {
    std::variant<std::monostate, int, std::string> wartosc;
    std::cout << std::boolalpha << std::holds_alternative<std::monostate>(wartosc) << '\n'; // true

    wartosc = 0;
    std::cout << std::holds_alternative<int>(wartosc) << '\n'; // true
}


true
true


`std::monostate` to prosty typ bez danych, którym możesz oznaczyć „brak podanej wartości”. Wybierasz to znaczenie w swoim programie. W pierwszej chwili wariant przechowuje właśnie `monostate`; po przypisaniu zera przechowuje `int`.

**Wariant z `monostate` nadal ma aktywną alternatywę.** Jest nią `monostate`, w tym przykładzie o indeksie 0. To ważne rozróżnienie względem stanu po wyjątku z następnego podrozdziału.

<a id="sekcja-06"></a>

### Gdy typ wymaga argumentu konstruktora

Struktura `NoDefault` wymaga podania liczby przy tworzeniu obiektu. Poniższa komórka pokazuje jej definicję i użycie z `monostate`:


In [3]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

struct NoDefault {
    explicit NoDefault(int number) : value(number) {}
    int value;
};

int main() {
    std::variant<std::monostate, NoDefault> wartosc;
    wartosc.emplace<NoDefault>(7);
    std::cout << std::get<NoDefault>(wartosc).value << '\n'; // 7
}


7


Konstruktor `NoDefault(int number)` zapisuje przekazaną liczbę w polu `value`. `explicit` wymaga jawnego użycia tego konstruktora przy konwersji do `NoDefault`.

Deklaracja wariantu bez wartości początkowej działa, ponieważ pierwszym typem jest `monostate`, który można utworzyć bez argumentów. Dopiero `emplace<NoDefault>(7)` tworzy właściwy obiekt z liczbą 7 i zastępuje nim `monostate`.

Gdyby `NoDefault` był pierwszym typem, trzeba byłoby od razu podać argument jego konstruktora, np. przez `std::in_place_type<NoDefault>`:


In [4]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

struct NoDefault {
    explicit NoDefault(int number) : value(number) {}
    int value;
};

int main() {
    std::variant<NoDefault, std::string> wartosc{std::in_place_type<NoDefault>, 7};
    std::cout << std::get<NoDefault>(wartosc).value << '\n'; // 7
}


7


<a id="sekcja-07"></a>

### Czy ten sam typ może wystąpić dwa razy?

Tak, ale wtedy sama nazwa typu nie wystarcza do wybrania miejsca na liście:


In [5]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

int main() {
    std::variant<int, int> wartosc{std::in_place_index<1>, 9};
    std::cout << wartosc.index() << '\n';      // 1
    std::cout << std::get<1>(wartosc) << '\n'; // 9
}


1
9


`std::in_place_index<1>` oznacza: „utwórz wartość w alternatywie o indeksie 1”, czyli w drugim `int`. Liczba 9 jest wartością, którą tam zapisujemy.

Próba `std::get<int>(wartosc)` byłaby błędem kompilacji: kompilator nie wie, o który z dwóch `int` chodzi. Taki sam problem dotyczy `get_if<int>` i `holds_alternative<int>`. Można używać `get<0>` i `get<1>`, a do sprawdzania wybranej pozycji - `index()`.

Jeśli dwie liczby mają różne znaczenie, lepiej wyrazić je w nazwach typów:


In [6]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

struct UserId {
    int value;
};
struct OrderId {
    int value;
};

int main() {
    std::variant<UserId, OrderId> identyfikator{OrderId{9}};
    std::cout << std::get<OrderId>(identyfikator).value << '\n'; // 9
}


9


Teraz widać, że odczytujemy identyfikator zamówienia, a nie dowolną liczbę z pozycji 1.


<a id="sekcja-08"></a>

## 2. Co oznacza `valueless_by_exception()`?

Zwykle wariant zawiera jedną z dozwolonych wartości. Istnieje jednak szczególny przypadek: podczas zmiany wartości może wystąpić wyjątek, po którym **wariant nie zawiera żadnej alternatywy**.

Wyobraź sobie następujący przebieg:

1. Wariant przechowuje liczbę 42.
2. Program chce zastąpić ją obiektem innego typu.
3. Dotychczasowa liczba zostaje usunięta.
4. Tworzenie nowego obiektu kończy się wyjątkiem.
5. Nowego obiektu nie udało się utworzyć, a starej liczby już nie ma.

Metoda `valueless_by_exception()` pozwala sprawdzić, czy wariant jest w takim stanie. Zwraca wtedy `true`.

**Nie każdy wyjątek powoduje utratę wartości.** Powyższy przebieg tłumaczy problem; dokładny skutek zależy od wykonywanej operacji i typów. Samo wywołanie `get` z niewłaściwym typem rzuca wyjątek, ale nie usuwa przechowywanej wartości.

<a id="sekcja-09"></a>

### Jak wymuszamy ten przypadek w demonstracji?

W poniższej komórce typ `ThrowsOnMove` ma konstruktor przenoszący, który celowo rzuca wyjątek. Konstruktor przenoszący służy do tworzenia obiektu na podstawie innego obiektu, którego zasoby można przejąć. Zapis `noexcept(false)` jawnie zaznacza, że ten konstruktor może rzucać wyjątki.

Program przygotowuje dwa warianty:


In [7]:
%%cpp
#include <iostream>
#include <string>
#include <variant>
#include <stdexcept>
#include <utility>

struct ThrowsOnMove {
    ThrowsOnMove() = default;
    ThrowsOnMove(const ThrowsOnMove&) = default;
    ThrowsOnMove& operator=(const ThrowsOnMove&) = default;
    ThrowsOnMove(ThrowsOnMove&&) noexcept(false) {
        throw std::runtime_error{"move failed"};
    }
    ThrowsOnMove& operator=(ThrowsOnMove&&) = default;
};

int main() {
    using Value = std::variant<int, ThrowsOnMove>;
    Value target{42};
    Value source{std::in_place_index<1>};

    try {
        target = std::move(source);
    } catch (const std::runtime_error&) {
        std::cout << "wyjatek przy przenoszeniu\n";
    }
    std::cout << std::boolalpha << target.valueless_by_exception() << '\n'; // true
}


wyjatek przy przenoszeniu
true


Definicja `ThrowsOnMove` znajduje się w tej samej komórce, przed `main()`. `target` początkowo zawiera liczbę, a `source` - obiekt `ThrowsOnMove`. `in_place_index<1>` tworzy obiekt źródłowy bezpośrednio w drugiej alternatywie, bez wcześniejszego przenoszenia go z innego miejsca.

`std::move(source)` pozwala wybrać operację przenoszącą; samo wywołanie `std::move` jeszcze niczego nie przenosi. Przenoszenie odbywa się w przypisaniu. Ponieważ warianty zawierają różne typy, `target` musi utworzyć u siebie obiekt `ThrowsOnMove`. Jego konstruktor rzuca wyjątek.

**Dla tego konkretnego przypadku standard gwarantuje, że `target` zostanie bez wartości:** chodzi o wyjątek z konstrukcji przenoszącej podczas przypisania przenoszącego między wariantami o różnych aktywnych indeksach.

<a id="sekcja-10"></a>

### Co można wtedy zrobić?

Obiekt wariantu nadal istnieje, ale nie ma w nim wartości, którą można odczytać:

- `index()` zwraca specjalną stałą `std::variant_npos`, zamiast indeksu typu.
- `get_if` zwraca `nullptr`, a `holds_alternative` zwraca `false` dla każdego typu z listy.
- `get` i `visit` rzucają `std::bad_variant_access`.

Możesz nadać mu nową wartość:


In [8]:
%%cpp
#include <iostream>
#include <string>
#include <variant>
#include <stdexcept>
#include <utility>

struct ThrowsOnMove {
    ThrowsOnMove() = default;
    ThrowsOnMove(const ThrowsOnMove&) = default;
    ThrowsOnMove& operator=(const ThrowsOnMove&) = default;
    ThrowsOnMove(ThrowsOnMove&&) noexcept(false) {
        throw std::runtime_error{"move failed"};
    }
    ThrowsOnMove& operator=(ThrowsOnMove&&) = default;
};

int main() {
    using Value = std::variant<int, ThrowsOnMove>;
    Value target{42};
    Value source{std::in_place_index<1>};

    try {
        target = std::move(source);
    } catch (const std::runtime_error&) {
        std::cout << "wyjatek przy przenoszeniu\n";
    }
    std::cout << std::boolalpha << target.valueless_by_exception() << '\n'; // true

    target = 10;
    std::cout << std::get<int>(target) << '\n';                             // 10
    std::cout << std::boolalpha << target.valueless_by_exception() << '\n'; // false
}


wyjatek przy przenoszeniu
true
10
false


Ta komórka najpierw samodzielnie odtwarza utratę wartości po wyjątku, a następnie przypisuje liczbę 10. Udane przypisanie przywraca zwykły stan wariantu. Można do tego użyć również udanego `emplace`.

Różnica do zapamiętania: **`monostate` oznacza wybraną przez programistę wartość „brak danych”, a `valueless_by_exception` oznacza brak jakiejkolwiek przechowywanej wartości po nieudanej operacji.**

Demonstracja sprawdza te warunki także przez `assert`. To instrukcja kontrolna: jeśli warunek jest fałszywy, program zostaje przerwany. Flaga `-DNDEBUG` wyłącza takie kontrole, więc nie używaj jej przy sprawdzaniu tego przykładu.


<a id="sekcja-11"></a>

## 3. Skąd wariant wie, który typ wybrać?

Przy prostym przypisaniu wybór wygląda naturalnie:


In [9]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

int main() {
    std::variant<int, double> liczba{3};
    std::cout << liczba.index() << '\n'; // 0: int
    liczba = 3.5;
    std::cout << liczba.index() << '\n'; // 1: double
}


0
1


Literał `3` ma typ `int`, a `3.5` ma typ `double`. W tych przypadkach jest jednoznaczne dopasowanie do typu z listy.

Trudniej jest wtedy, gdy podana wartość wymaga konwersji. Konwersja to zamiana wartości jednego typu na inny, np. liczby całkowitej na zmiennoprzecinkową. Wariant korzysta z reguł wyboru przeciążenia i ograniczeń konstruktora, aby ustalić, którą alternatywę utworzyć. Nie wybiera po prostu pierwszego typu z listy. Jeśli nie ma poprawnego, jednoznacznego wyboru, kod nie zostanie skompilowany.

Możesz wskazać typ bezpośrednio:


In [10]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

int main() {
    std::variant<int, double> liczba{std::in_place_type<double>, 3};
    std::cout << liczba.index() << '\n';           // 1
    std::cout << std::get<double>(liczba) << '\n'; // 3
}


1
3


`std::in_place_type<double>` oznacza: „utwórz alternatywę `double`, używając dalszych argumentów”. Zapisana wartość to `3.0` typu `double`, chociaż `std::cout` domyślnie wypisuje ją jako `3`.

<a id="sekcja-12"></a>

### Dlaczego w przykładach tekst zapisujemy jawnie?

Literał `"abc"` i obiekt `std::string{"abc"}` to różne typy. W drugim zapisie jawnie tworzysz obiekt tekstowy:


In [11]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

int main() {
    std::variant<bool, std::string> tekst{std::string{"abc"}};
    std::cout << std::get<std::string>(tekst) << '\n'; // abc
}


abc


Ma to znaczenie m.in. dla historycznej pułapki: przy inicjalizacji wariantu z alternatywami `bool` i `std::string` stara biblioteka mogła wybrać `bool` dla literału tekstowego. Adres tekstu można bowiem przekształcić na wartość logiczną. Nowsze reguły ograniczyły takie niepożądane konwersje.

Szczegóły opisuje dokument [P0608R3](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2018/p0608r3.html) oraz [opis konstruktora z listą poprawek](https://en.cppreference.com/w/cpp/utility/variant/variant). Poprawki trafiają również do bibliotek pracujących w trybie C++17, więc sama flaga `-std=c++17` nie oznacza identycznego zachowania każdej historycznej wersji biblioteki.

Praktyczna zasada: jeśli chcesz uniknąć wątpliwości, podaj wartość zamierzonego typu albo użyj `in_place_type`.


<a id="sekcja-13"></a>

## 4. Ile pamięci zajmuje wariant i czy można go kopiować?

<a id="sekcja-14"></a>

### Miejsce musi wystarczyć na każdy dozwolony typ

Dla `std::variant<int, double, std::string>` ten sam obiekt musi umieć pomieścić każdy z trzech typów. Nie musi przechowywać ich jednocześnie, ale potrzebuje dość miejsca na największy z nich oraz informacji, który jest aktualnie wybrany.

Dlatego wariant zawierający małe `int` nie staje się mniejszy tylko dlatego, że obecnie nie przechowuje tekstu. Rozmiar obiektu jest ustalony przez jego typ.

Na końcu komórki demonstracyjnej poniżej wypisujemy:

- `sizeof(...)` - rozmiar obiektu w bajtach;
- `alignof(...)` - wymagane wyrównanie, czyli ograniczenie dotyczące adresów, pod którymi obiekt może być umieszczony w pamięci.

Przy wyrównaniu 8 adres obiektu musi być wielokrotnością 8. Wyrównanie może powodować pozostawienie dodatkowych bajtów w układzie obiektu. Z tego powodu nie wystarczy dodać „rozmiar największego typu + jeden bajt na numer”. Standard nie narzuca konkretnego układu ani rozmiaru znacznika.

W naszym sprawdzeniu wariant `int, double, std::string` zajmował 32 bajty w libc++ i 40 w libstdc++. To wyniki konkretnych środowisk, a nie wartości wymagane na każdym komputerze.

<a id="sekcja-15"></a>

### Czy wariant używa pamięci dynamicznej?

Przechowywana alternatywa znajduje się wewnątrz obiektu wariantu. Sam wariant nie potrzebuje osobnej alokacji na tę alternatywę. Jednak jej własne dane mogą korzystać z pamięci dynamicznej - np. obiekt `std::string` może przechowywać długi tekst poza swoim obiektem.

Dlatego „obiekt tekstowy mieści się wewnątrz wariantu” nie oznacza, że wszystkie znaki tekstu również muszą się tam mieścić.

<a id="sekcja-16"></a>

### Od czego zależy kopiowanie?

Aby skopiować wariant, trzeba móc skopiować każdą z jego alternatyw. Kompilator musi dopuścić kopiowanie całego typu niezależnie od tego, która alternatywa jest akurat aktywna.

Jeśli na liście znajduje się `std::unique_ptr<int>`, wariantu nie można kopiować, ponieważ `unique_ptr` nie pozwala kopiować własności zasobu. Taki wariant może natomiast pozwalać na przenoszenie. Podobnie dostępność przypisania zależy od operacji udostępnianych przez typy alternatyw.

Jako alternatywy nie można użyć bezpośrednio referencji, np. `int&`, typu `void` ani surowej tablicy. Gdy potrzebujesz odwołania do istniejącego obiektu, jednym z rozwiązań jest `std::reference_wrapper`. Nie przejmuje on jednak własności obiektu: trzeba dopilnować, aby wskazywany obiekt nadal istniał.

Duże alternatywy zwiększają rozmiar każdego wariantu danego typu. Nie zakładaj też, że `variant` zawsze będzie szybszy od funkcji wirtualnych - wydajność należy sprawdzić dla konkretnego programu.


<a id="sekcja-17"></a>

### Demonstracja: Pusty stan, wyjątek i rozmiar wariantu

Ta komórka łączy przypadki z podrozdziałów 1-4: `monostate`, powtórzone typy, utratę wartości po wyjątku, odzyskanie wartości oraz pomiary pamięci.


In [12]:
%%cpp
// Cel: pusty stan dziedzinowy, duplikaty i gwarantowany przypadek utraty wartości.
#include <cassert>
#include <iostream>
#include <stdexcept>
#include <string>
#include <utility>
#include <variant>

struct NoDefault {
    explicit NoDefault(int number) : value(number) {}
    int value;
};

struct ThrowsOnMove {
    ThrowsOnMove() = default;
    ThrowsOnMove(const ThrowsOnMove&) = default;
    ThrowsOnMove& operator=(const ThrowsOnMove&) = default;
    ThrowsOnMove(ThrowsOnMove&&) noexcept(false) {
        throw std::runtime_error{"move failed"};
    }
    ThrowsOnMove& operator=(ThrowsOnMove&&) = default;
};

int main() {
    std::variant<std::monostate, NoDefault> maybe;
    assert(std::holds_alternative<std::monostate>(maybe));
    std::cout << "monostate: " << maybe.index() << '\n';
    maybe.emplace<NoDefault>(7);
    std::cout << "NoDefault: " << std::get<NoDefault>(maybe).value << '\n';

    std::variant<int, int> repeated{std::in_place_index<1>, 9};
    std::cout << "duplikaty: index=" << repeated.index() << ", value=" << std::get<1>(repeated)
              << '\n';

    using Value = std::variant<int, ThrowsOnMove>;
    Value target{42};
    Value source{std::in_place_index<1>};
    try {
        // Różne indeksy: przypisanie przenoszące konstruuje nową alternatywę
        target = std::move(source);
    } catch (const std::runtime_error&) {
        std::cout << "wyjatek przy przenoszeniu\n";
    }
    assert(target.valueless_by_exception());
    assert(target.index() == std::variant_npos);
    assert(std::get_if<int>(&target) == nullptr);
    std::cout << std::boolalpha << "valueless: " << target.valueless_by_exception() << '\n';
    try {
        std::visit(
            [](const auto&) {
            },
            target
        );
    } catch (const std::bad_variant_access&) {
        std::cout << "visit: bad_variant_access\n";
    }
    target = 10;
    assert(!target.valueless_by_exception());
    std::cout << "po naprawie: " << std::get<int>(target) << '\n';

    using Stored = std::variant<int, double, std::string>;
    static_assert(sizeof(Stored) >= sizeof(int));
    static_assert(sizeof(Stored) >= sizeof(double));
    static_assert(sizeof(Stored) >= sizeof(std::string));
    static_assert(alignof(Stored) >= alignof(std::string));
    std::cout << "sizeof wariantu: " << sizeof(Stored) << '\n';
    std::cout << "sizeof string: " << sizeof(std::string) << '\n';
    std::cout << "alignof wariantu: " << alignof(Stored) << '\n';
}


monostate: 0
NoDefault: 7
duplikaty: index=1, value=9
wyjatek przy przenoszeniu
valueless: true
visit: bad_variant_access
po naprawie: 10
sizeof wariantu: 32
sizeof string: 24
alignof wariantu: 8


**Jak czytać wynik:** Program zestawia prawidłowy pusty stan `monostate`, rozróżnianie powtórzonych typów i szczególny przypadek utraty wartości po wyjątku. Końcowe trzy pomiary pamięci dotyczą bieżącej platformy i biblioteki; nie są stałymi określonymi przez standard.


<a id="sekcja-18"></a>

## 5. Kiedy wybrać inne rozwiązanie i czego uczyć się dalej?

<a id="sekcja-19"></a>

### Dobierz narzędzie do tego, co chcesz opisać

`variant` przydaje się, gdy znasz listę różnych możliwości, np. „liczba albo tekst” lub „rozłączony albo łączenie albo połączony”. Wybór jednej z możliwości nazywa się też *sum type* - to nazwa modelu danych, a nie dodatkowa funkcja C++.

Inne sytuacje mogą wymagać prostszego lub innego narzędzia:

- **Liczba albo brak liczby:** `std::optional<int>`. Wszystkie obecne wartości mają jeden typ `int`.
- **Imię i wiek jednocześnie:** zwykła struktura z dwoma polami. Obie informacje są potrzebne razem.
- **Przechowanie wartości bez wpisywania listy typów w deklaracji pojemnika:** `std::any`. Obsługuje typy kopiowalne; aby odzyskać wartość, musisz znać jej typ.
- **Niskopoziomowe współdzielenie pamięci przez składowe:** `union`. Sam nie pamięta, która składowa jest aktywna, i nie zapewnia kontroli odczytu takiej jak `variant`.
- **Rodzina obiektów rozszerzana o kolejne typy w niezależnych częściach programu:** warto rozważyć wspólny interfejs i funkcje wirtualne. Wariant wymaga podania listy alternatyw w jego typie.

<a id="sekcja-20"></a>

### Co zmieniają nowsze standardy?

Cały uruchamialny kod w tych materiałach korzysta z C++17. Od C++20 można m.in. użyć `std::visit<R>`, aby wskazać typ wyniku odwiedzania. `R` oznacza tutaj wybrany przez programistę typ, np. `double`. W przykładach pozostajemy przy zapisie dostępnym w C++17. Przed użyciem rozszerzeń sprawdź obsługę w swojej bibliotece i kompilatorze.

<a id="sekcja-21"></a>

### Dalszy temat: drzewo wyrażeń

Wyrażenie `2 + 3` można opisać jako węzeł dodawania z dwoma podwyrażeniami. Każde podwyrażenie może być liczbą albo kolejnym działaniem. `variant` pasuje do tego wyboru, ale pojawia się problem rekurencji: węzeł nie może zawierać w sobie kolejnego pełnego obiektu tego samego typu bez końca.

Jednym z rozwiązań jest przechowywanie podwęzłów przez inteligentne wskaźniki. Węzeł przechowuje wtedy wskaźniki do osobnych obiektów zamiast umieszczać całą rekurencyjną strukturę bezpośrednio w swoim wnętrzu. To temat do dalszej nauki, wymagający także zrozumienia własności i czasu życia obiektów.

<a id="sekcja-22"></a>

## Dobre praktyki do zapamiętania

- Wybieraj alternatywy opisujące znaczenie danych; nazwane struktury pomagają odróżnić podobne przypadki.
- Używaj `get_if`, gdy niezgodna alternatywa jest oczekiwaną sytuacją; `get`, gdy chcesz odczytu z kontrolą przez wyjątek.
- Stosuj `visit` do wspólnej operacji na alternatywach. Przejrzyj wszystkie przypadki po zmianie listy typów.
- Nie przechowuj wskaźników do alternatywy dłużej niż trwa jej życie.
- Odróżniaj pusty stan modelu (`monostate`) od braku wartości po wyjątku.
- Nie dodawaj alternatyw „na zapas”. Zbyt szeroki wariant utrudnia opis reguł programu.

<a id="sekcja-23"></a>

## Dodatek: rozpoznawanie błędów kompilacji

Poniższe dwa programy są **celowo błędne**. Można je wykonać tak jak pozostałe komórki. Opcja `--expect-error` oznacza, że odrzucenie kodu przez kompilator jest tutaj oczekiwanym wynikiem. Po zastosowaniu opisanej poprawki usuń `--expect-error`, aby skompilować i uruchomić poprawiony program.

Pierwszy: typ `int` występuje dwa razy, więc nie można wybrać go przez sam typ. Poprawka: użyć `std::get<1>(value)`.


In [13]:
%%cpp --expect-error
#include <variant>
int main() {
    std::variant<int, int> value{std::in_place_index<1>, 9};
    (void)std::get<int>(value); // BLAD: typ nie jest jednoznaczny
}


Oczekiwany błąd kompilacji. Początek komunikatu kompilatora:
In file included from example.cpp:1:
In file included from /Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/variant:230:
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/__tuple/find_index.h:45:17: error: static assertion failed due to requirement 'value != __ambiguous': type occurs more than once in type list
   45 |   static_assert(value != __ambiguous, "type occurs more than once in type list");
      |                 ^~~~~~~~~~~~~~~~~~~~
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/__tuple/find_index.h:56:38: note: in instantiation of template class 'std::__find_detail::__find_exactly_one_checked<int, int, int>' requested here
   56 | struct __find_exactly_one_t : public __find_detail::__find_exactly_one_checked<_T1, _Args...> {};
      |                                      ^
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/varia

Drugi: gałęzie odwiedzającego mają różne typy wyniku. Poprawka: dopisać `-> double` do lambdy, aby obie instancje zwracały `double`.


In [14]:
%%cpp --expect-error
#include <type_traits>
#include <variant>
int main() {
    std::variant<int, double> value{1};
    (void)std::visit(
        [](auto x) {
            if constexpr (std::is_same_v<decltype(x), int>) {
                return 1;
            } else {
                return 1.5;
            }
        },
        value
    ); // BŁAD: wynik int albo double
}


Oczekiwany błąd kompilacji. Początek komunikatu kompilatora:
In file included from example.cpp:2:
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/variant:520:9: error: static assertion failed due to requirement '__all<false>::value': `std::visit` requires the visitor to have a single return type.
  520 |         __all<is_same_v<_Fp, _Fs>...>::value, "`std::visit` requires the visitor to have a single return type.");
      |         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/c++/v1/variant:525:5: note: in instantiation of function template specialization 'std::__variant_detail::__visitation::__base::__std_visit_visitor_return_type_check<int (*)(std::__variant_detail::__visitation::__variant::__value_visitor<(lambda at example.cpp:6:9)> &&, std::__variant_detail::__base<std::__variant_detail::_Trait::_TriviallyAvailable, int, double> &), double (*)(std::__variant_detail::__visitation::__variant::__value_visi

<a id="sekcja-24"></a>

## Artykuły i dokumentacja

<a id="sekcja-25"></a>

### Artykuły z przykładami

- [C++ Stories: Everything You Need to Know About std::variant](https://www.cppstories.com/2018/06/variant/) - wprowadzenie i praktyczne zastosowania.
- [C++ Stories: Runtime Polymorphism with std::variant and std::visit](https://www.cppstories.com/2020/04/variant-virtual-polymorphism.html/) - porównanie z funkcjami wirtualnymi.

<a id="sekcja-26"></a>

### Dokumentacja konkretnych operacji

- [cppreference: std::monostate](https://en.cppreference.com/cpp/utility/variant/monostate) - jawny pusty stan modelu.
- [cppreference: valueless_by_exception](https://en.cppreference.com/cpp/utility/variant/valueless_by_exception) - brak wartości po wyjątku.
- [cppreference: Konstruktory std::variant](https://en.cppreference.com/cpp/utility/variant/variant) - wybór alternatywy i konstrukcja in_place.
- [cppreference: std::visit (funkcja wolna, od C++17)](https://en.cppreference.com/cpp/utility/variant/visit2) - wywoływanie odwiedzającego dla jednego lub wielu wariantów.

W artykułach i dokumentacji występują także rozszerzenia późniejszych standardów. Przykłady tego rozdziału pozostają w C++17.


[Poprzednia część](03_zastosowania.ipynb) | [Start](00_start.ipynb) | [Pełny przewodnik](../PRZEWODNIK.md) | [Ściąga](../SCIAGA.md)
